# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import VotingClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, cross_val_predict

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)


def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)


def voting_cross_val_predict(probas: list[np.array]) -> np.array:

    weights = np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
    weights /= weights.sum()

    voting_proba = np.sum(proba*weight for proba, weight in zip(estimators, weights))

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test.parquet')

In [5]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


In [6]:
X_test.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.070991
439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,...,0.298107,0.185508,0.178575,0.147360,0.298107,0.298107,0.321361,0.178575,0.147360,0.253721
439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,...,0.298107,0.207392,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.070991
439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,...,0.102471,0.207392,0.178575,0.279021,0.102471,0.102471,0.071042,0.178575,0.279021,0.216120
439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,...,0.298107,0.207392,0.178575,0.111582,0.298107,0.298107,0.206543,0.178575,0.111582,0.216120


# Machine Learning

In [7]:
lgbm = load_pickle('../models/model_lightgbm.pkl')
cat = load_pickle('../models/model_catboost.pkl')
xgb = load_pickle('../models/model_xgboost.pkl')
hist = load_pickle('../models/model_hist_gradient_boosting.pkl')
knn = load_pickle('../models/model_knn.pkl')
lg = load_pickle('../models/model_logistic_regression.pkl')

## Train Dataset

In [8]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

lgbm_proba = cross_val_predict(lgbm, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
cat_proba = cross_val_predict(cat, X_train, y_train.PitNextLap, cv=cv, n_jobs=2, method='predict_proba')[:, 1]
xgb_proba = cross_val_predict(xgb, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
hist_proba = cross_val_predict(hist, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
lg_proba = cross_val_predict(lg, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba')[:, 1]
knn_proba = cross_val_predict(knn, X_train, y_train.PitNextLap, cv=cv, n_jobs=-1, method='predict_proba',)[:, 1]

In [9]:
X_train_stacking = pd.DataFrame({
    'lgbm_proba': lgbm_proba,
    'cat_proba': cat_proba,
    'xgb_proba': xgb_proba,
    'hist_proba': hist_proba,
    'lg_proba': lg_proba,
    'knn_proba': knn_proba
})

# Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({
    'lgbm_proba': lgbm.predict_proba(X_test)[:, 1],
    'cat_proba': cat.predict_proba(X_test)[:, 1],
    'xgb_proba': xgb.predict_proba(X_test)[:, 1],
    'hist_proba': hist.predict_proba(X_test)[:, 1],
    'lg_proba': lg.predict_proba(X_test)[:, 1],
    'knn_proba': knn.predict_proba(X_test)[:, 1]
})

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


# Saving

In [11]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking.parquet')

In [16]:
X_train_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043
